# KL-IG: 8-Method Evaluation (3σ variants + 5 baselines)
ResNet50, 100 images, 7 core metrics

In [ ]:
!git clone https://github.com/Shameen5375/KLIG_V1.git 2>/dev/null || echo "Repo already cloned"
!pip install -q captum datasets opencv-python-headless


In [ ]:
import os, sys, math, json, pickle, warnings
from pathlib import Path
from collections import defaultdict
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torchvision.models import ResNet50_Weights, resnet50
from scipy import stats
from tqdm.auto import tqdm
import cv2

ROOT = Path.cwd()
for candidate in [ROOT, ROOT / "infocube-main",
                  Path("/content/KLIG_V1/infocube-main"),
                  Path("/content/KLIG_V1")]:
    if (candidate / "klig").exists():
        ROOT = candidate
        break
sys.path.append(str(ROOT))

from klig.image.attribution import ImageAttributor
from klig.image.stopping import find_sigma_stop
from klig.compare.captum_baselines import (
    run_ig, run_smoothgrad, run_expected_gradients, _absmax_collapse,
)
from captum.attr import IntegratedGradients, Saliency

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Root:   {ROOT}")


In [ ]:
N_IMGS = 100
N_STEPS = 50
N_SAMPLES = 10
TIER_A = 100
BLUR_SIGMA = 16.0
BLUR_KERNEL = 51
IG_STEPS = 50
EG_SAMPLES = 50
SG_SAMPLES = 50

N_INSERTION_STEPS = 50
N_SENS_SUBSETS = 30
SENS_FRACTIONS = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.8]
PERTURBATION_SIGMAS = [0.01, 0.02, 0.05, 0.1, 0.2]
PERTURBATION_RUNS = 3
ROB_EPS = 0.02
ROB_TRIALS = 5
OCCLUSION_PATCH = 14
OCCLUSION_STRIDE = 7
OCCLUSION_RATIOS = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
TRANSFORM = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

HF_DATASET_NAME = "evanarlian/imagenet_1k_resized_256"
HF_SPLIT = "val"

# 8 methods: 3 KL-IG σ variants + 5 baselines
methods_all = [
    "KL-IG (adapt)",
    "KL-IG (cap=1)",
    "KL-IG (σ=0.25)",
    "IDG",
    "ExpGrad",
    "IG-zero",
    "SmoothGrad",
    "Vanilla Grad",
]

COLORS_ALL = {
    "KL-IG (adapt)":  "#1B5E3F",
    "KL-IG (cap=1)":  "#2E8B57",
    "KL-IG (σ=0.25)": "#3CB371",
    "IDG":            "#E07B39",
    "ExpGrad":        "#DC143C",
    "IG-zero":        "#7B68EE",
    "SmoothGrad":     "#1E90FF",
    "Vanilla Grad":   "#8B4513",
}

print(f"N_IMGS={N_IMGS}, methods={len(methods_all)}")


In [ ]:
def load_model():
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights).to(DEVICE).eval()
    return model, weights.meta["categories"]

def denormalize(x):
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(-1, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=x.device).view(-1, 1, 1)
    if x.dim() == 4:
        mean, std = mean.unsqueeze(0), std.unsqueeze(0)
    return (x * std + mean).clamp(0, 1)

def make_blur_baseline(x, kernel_size=BLUR_KERNEL, sigma=BLUR_SIGMA):
    coords = torch.arange(kernel_size, dtype=torch.float32, device=x.device) - kernel_size // 2
    k1d = torch.exp(-0.5 * (coords / sigma) ** 2)
    k1d = k1d / k1d.sum()
    kh = k1d.view(1, 1, -1, 1).expand(3, -1, -1, -1)
    kw = k1d.view(1, 1, 1, -1).expand(3, -1, -1, -1)
    pad = kernel_size // 2
    out = F.conv2d(x, kh, padding=(pad, 0), groups=3)
    return F.conv2d(out, kw, padding=(0, pad), groups=3)

def make_eg_background(x, n=50):
    return torch.randn(n, *x.squeeze(0).shape, device=x.device)

model, imagenet_labels = load_model()
print(f"Model: ResNet50, {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")


In [ ]:
def load_imagenet_subset(n_images):
    try:
        from datasets import load_dataset
        print(f"[dataset] HuggingFace {HF_DATASET_NAME} [{HF_SPLIT}]")
        ds = load_dataset(HF_DATASET_NAME, split=HF_SPLIT, streaming=True)
        seen, out = set(), []
        for ex in ds:
            y = int(ex["label"])
            if y in seen:
                continue
            seen.add(y)
            out.append((ex["image"].convert("RGB"), y))
            if len(out) >= n_images:
                break
        return out
    except Exception as e:
        print(f"[dataset] Failed: {e}")
        return []

raw_samples = load_imagenet_subset(TIER_A)
dataset = []
with torch.no_grad():
    for i, (pil, gt) in enumerate(tqdm(raw_samples, desc="prep")):
        x = TRANSFORM(pil).unsqueeze(0).to(DEVICE)
        probs = model(x).softmax(-1)[0]
        top1 = int(probs.argmax())
        target = gt if probs[gt].item() > 0.05 else top1
        dataset.append({
            "idx": i,
            "x": x,
            "target": target,
            "label_str": imagenet_labels[target],
        })

results = {}
for row in dataset:
    results[row["idx"]] = row

eval_idx = list(range(min(N_IMGS, len(dataset))))
print(f"Dataset: {len(dataset)} images, evaluating {len(eval_idx)}")


In [ ]:
def get_sigma_final(sigma_name, model, x, target):
    if sigma_name == "adapt":
        return max(find_sigma_stop(model, x, target=target, tau=0.95), 1.0/256.0)
    elif sigma_name == "cap=1":
        return min(max(find_sigma_stop(model, x, target=target, tau=0.95), 1.0/256.0), 1.0)
    elif sigma_name == "σ=0.25":
        return 0.25

def compute_klig(model, x, target, sigma_final):
    attr = ImageAttributor(model=model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                           sigma_final=sigma_final, device=DEVICE)
    res = attr.attribute(x, target=target, show_progress=False)
    return res, res.attr_map("absmax")

def compute_ig_zero(model, x, target):
    return run_ig(model, x, target, n_steps=IG_STEPS)

def compute_idg(model, x, target):
    x_inp = x.clone().detach().requires_grad_(True)
    model.zero_grad()
    out = model(x_inp)
    out[0, target].backward()
    grad = x_inp.grad
    if grad is None:
        return torch.zeros(x.shape[-2:], device=x.device)
    attr = x_inp.detach() * grad.detach()
    return _absmax_collapse(attr)

def compute_expgrad(model, x, target):
    bg = make_eg_background(x, n=EG_SAMPLES)
    return run_expected_gradients(model, x, target, background=bg, n_samples=EG_SAMPLES)

def compute_smoothgrad(model, x, target):
    return run_smoothgrad(model, x, target, n_samples=SG_SAMPLES)

def compute_vanilla_grad(model, x, target):
    sal = Saliency(model)
    attr = sal.attribute(x, target=target, abs=False)
    return _absmax_collapse(attr.detach())

COMPUTE_FN = {
    "IDG": compute_idg,
    "ExpGrad": compute_expgrad,
    "IG-zero": compute_ig_zero,
    "SmoothGrad": compute_smoothgrad,
    "Vanilla Grad": compute_vanilla_grad,
}

print("Attribution functions ready")


In [ ]:
def sensitivity_n(model, x, attr_map, target, fractions=SENS_FRACTIONS, n_subsets=N_SENS_SUBSETS):
    H, W = attr_map.shape
    n_pix = H * W
    attr_flat = attr_map.detach().cpu().numpy().ravel()
    with torch.no_grad():
        f_orig = model(x).softmax(-1)[0, target].item()
    pccs = []
    for frac in fractions:
        n = max(1, int(frac * n_pix))
        attr_sums, output_diffs = [], []
        for _ in range(n_subsets):
            subset = np.random.choice(n_pix, size=n, replace=False)
            attr_sums.append(attr_flat[subset].sum())
            x_masked = x.clone()
            hi, wi = subset // W, subset % W
            x_masked[0, :, hi, wi] = 0.0
            with torch.no_grad():
                f_masked = model(x_masked).softmax(-1)[0, target].item()
            output_diffs.append(f_orig - f_masked)
        r, _ = stats.pearsonr(attr_sums, output_diffs)
        pccs.append(r if not np.isnan(r) else 0.0)
    return np.array(pccs)

def insertion_deletion(model, x, attr_map, target, substrate, n_steps=N_INSERTION_STEPS):
    H, W = attr_map.shape
    n_pix = H * W
    order = attr_map.detach().cpu().abs().view(-1).argsort(descending=True)
    pps = max(1, n_pix // n_steps)
    ins_img, del_img = substrate.clone(), x.clone()
    ins_scores, del_scores = [], []
    with torch.no_grad():
        ins_scores.append(model(ins_img).softmax(-1)[0, target].item())
        del_scores.append(model(del_img).softmax(-1)[0, target].item())
        for step in range(1, n_steps + 1):
            s, e = (step-1)*pps, min(step*pps, n_pix)
            if s >= n_pix:
                ins_scores.append(ins_scores[-1]); del_scores.append(del_scores[-1]); continue
            idx = order[s:e]
            hi, wi = idx // W, idx % W
            ins_img[0, :, hi, wi] = x[0, :, hi, wi]
            del_img[0, :, hi, wi] = substrate[0, :, hi, wi]
            ins_scores.append(model(ins_img).softmax(-1)[0, target].item())
            del_scores.append(model(del_img).softmax(-1)[0, target].item())
    ins = np.array(ins_scores); dl = np.array(del_scores)
    return float(np.trapz(ins, dx=1/n_steps)), float(np.trapz(dl, dx=1/n_steps)), ins, dl

def estimate_object_mask(x, attr_map):
    H, W = attr_map.shape
    a = np.abs(attr_map.detach().cpu().numpy())
    seed = (a >= np.percentile(a, 80)).astype(np.uint8)
    img_rgb = denormalize(x[0]).detach().cpu().permute(1, 2, 0).numpy()
    img_bgr = (img_rgb * 255).clip(0, 255).astype(np.uint8)[:, :, ::-1].copy()
    gc_mask = np.where(seed, cv2.GC_PR_FGD, cv2.GC_PR_BGD).astype(np.uint8)
    gc_mask[(a >= np.percentile(a, 95))] = cv2.GC_FGD
    edge_mask = np.zeros((H, W), dtype=np.uint8)
    border = max(H, W) // 10
    edge_mask[:border, :] = 1; edge_mask[-border:, :] = 1
    edge_mask[:, :border] = 1; edge_mask[:, -border:] = 1
    gc_mask[(edge_mask == 1) & (a < np.percentile(a, 10))] = cv2.GC_BGD
    try:
        bgd_model = np.zeros((1, 65), np.float64)
        fgd_model = np.zeros((1, 65), np.float64)
        cv2.grabCut(img_bgr, gc_mask, None, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_MASK)
        return np.where((gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD), 1, 0).astype(np.uint8)
    except:
        return seed

def object_focus_ratio(attr_map, obj_mask):
    a = np.abs(attr_map.detach().cpu().numpy())
    total = a.sum()
    return float(a[obj_mask == 1].sum() / total) if total > 1e-12 else 0.0

def gini_coefficient(attr_map):
    a = np.abs(attr_map.detach().cpu().numpy().ravel())
    a = a[a > 0]
    if len(a) == 0:
        return 0.0
    a = np.sort(a)
    n = len(a)
    return float((2 * np.sum(np.arange(1, n + 1) * a)) / (n * np.sum(a)) - (n + 1) / n)

def sanity_similarity(map_a, map_b):
    a = np.abs(map_a.detach().cpu().numpy().ravel())
    b = np.abs(map_b.detach().cpu().numpy().ravel())
    if a.std() < 1e-9 or b.std() < 1e-9:
        return 0.0
    rho, _ = stats.spearmanr(a, b)
    return 0.0 if np.isnan(rho) else float(rho)

print("Metric functions ready")


In [ ]:
# ── Metric 1: Sensitivity-n (All 8 Methods) ──

all_sens = {m: [] for m in methods_all}
all_sens_mean = {m: [] for m in methods_all}

for img_i, idx in enumerate(tqdm(eval_idx, desc="sensitivity-n")):
    r = results[idx]
    x, tgt = r["x"], r["target"]

    for sigma_name in ["adapt", "cap=1", "σ=0.25"]:
        sf = get_sigma_final(sigma_name, model, x, tgt)
        _, amap = compute_klig(model, x, tgt, sf)
        curve = sensitivity_n(model, x, amap, tgt)
        all_sens[f"KL-IG ({sigma_name})"].append(curve)
        all_sens_mean[f"KL-IG ({sigma_name})"].append(float(curve.mean()))

    for m in ["IDG", "ExpGrad", "IG-zero", "SmoothGrad", "Vanilla Grad"]:
        amap = COMPUTE_FN[m](model, x, tgt)
        curve = sensitivity_n(model, x, amap, tgt)
        all_sens[m].append(curve)
        all_sens_mean[m].append(float(curve.mean()))

fig, ax = plt.subplots(figsize=(13, 6))
fracs = np.array(SENS_FRACTIONS)
for m in methods_all:
    stacked = np.stack(all_sens[m])
    mu = stacked.mean(axis=0)
    ax.plot(fracs, mu, "o-", color=COLORS_ALL[m], lw=2, ms=4,
            label=f"{m} ({np.mean(all_sens_mean[m]):.3f})")
ax.axhline(0, color="gray", ls="--", alpha=0.5)
ax.set_xlabel("Fraction of pixels perturbed", fontsize=11)
ax.set_ylabel("PCC", fontsize=11)
ax.set_title(f"Sensitivity-n (↑)  n={len(eval_idx)}", fontweight="bold", fontsize=12)
ax.legend(fontsize=8, loc="best")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Metric 2: Insertion / Deletion (All 8 Methods) ──

all_ins_auc = {m: [] for m in methods_all}
all_del_auc = {m: [] for m in methods_all}
all_ins_curve = {m: [] for m in methods_all}
all_del_curve = {m: [] for m in methods_all}

for img_i, idx in enumerate(tqdm(eval_idx, desc="insertion/deletion")):
    r = results[idx]
    x, tgt = r["x"], r["target"]
    substrate = make_blur_baseline(x)

    for sigma_name in ["adapt", "cap=1", "σ=0.25"]:
        sf = get_sigma_final(sigma_name, model, x, tgt)
        _, amap = compute_klig(model, x, tgt, sf)
        ia, da, ic, dc = insertion_deletion(model, x, amap, tgt, substrate)
        all_ins_auc[f"KL-IG ({sigma_name})"].append(ia)
        all_del_auc[f"KL-IG ({sigma_name})"].append(da)
        all_ins_curve[f"KL-IG ({sigma_name})"].append(ic)
        all_del_curve[f"KL-IG ({sigma_name})"].append(dc)

    for m in ["IDG", "ExpGrad", "IG-zero", "SmoothGrad", "Vanilla Grad"]:
        amap = COMPUTE_FN[m](model, x, tgt)
        ia, da, ic, dc = insertion_deletion(model, x, amap, tgt, substrate)
        all_ins_auc[m].append(ia); all_del_auc[m].append(da)
        all_ins_curve[m].append(ic); all_del_curve[m].append(dc)

x_pos = np.arange(len(methods_all))
fig, ax = plt.subplots(figsize=(13, 5))
ins_means = [np.mean(all_ins_auc[m]) for m in methods_all]
del_means = [np.mean(all_del_auc[m]) for m in methods_all]
ax.bar(x_pos - 0.2, ins_means, 0.4, label="Insertion ↑", color=[COLORS_ALL[m] for m in methods_all], alpha=0.85)
ax.bar(x_pos + 0.2, del_means, 0.4, label="Deletion ↓", color=[COLORS_ALL[m] for m in methods_all], alpha=0.4, hatch="//")
ax.set_xticks(x_pos)
ax.set_xticklabels(methods_all, rotation=20, fontsize=9)
ax.set_ylabel("AUC", fontsize=11)
ax.set_title(f"Insertion/Deletion AUC  n={len(eval_idx)}", fontweight="bold", fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Metric 3: Occlusion Correlation vs Ratio (Line Plot) ──

import math

def ratio_to_patch(ratio, img_size=224):
    return max(2, int(round(math.sqrt(ratio * img_size * img_size))))

def occlusion_map(model, x, target, ps, stride=OCCLUSION_STRIDE):
    _, C, H, W = x.shape
    with torch.no_grad():
        f0 = model(x)[0, target].item()
    hs = list(range(0, H - ps + 1, stride))
    ws = list(range(0, W - ps + 1, stride))
    om = np.zeros((len(hs), len(ws)))
    with torch.no_grad():
        for i, h in enumerate(hs):
            for j, w in enumerate(ws):
                mk = x.clone(); mk[0, :, h:h+ps, w:w+ps] = 0
                om[i, j] = f0 - model(mk)[0, target].item()
    return om

def attr_to_grid(attr_map, ps, stride, H, W):
    a = attr_map.detach().cpu().abs().numpy()
    hs = list(range(0, H - ps + 1, stride))
    ws = list(range(0, W - ps + 1, stride))
    g = np.zeros((len(hs), len(ws)))
    for i, h in enumerate(hs):
        for j, w in enumerate(ws):
            g[i, j] = a[h:h+ps, w:w+ps].mean()
    return g

occ_by_ratio = {m: [] for m in methods_all}

for ratio in tqdm(OCCLUSION_RATIOS, desc="occlusion sweep"):
    ps = ratio_to_patch(ratio)
    occ_per_method = {m: [] for m in methods_all}

    for idx in eval_idx:
        r = results[idx]
        x, tgt = r["x"], r["target"]
        _, _, H, W = x.shape
        om = occlusion_map(model, x, tgt, ps=ps)

        for sigma_name in ["adapt", "cap=1", "σ=0.25"]:
            sf = get_sigma_final(sigma_name, model, x, tgt)
            _, amap = compute_klig(model, x, tgt, sf)
            ag = attr_to_grid(amap, ps, OCCLUSION_STRIDE, H, W)
            rho, _ = stats.spearmanr(om.ravel(), ag.ravel())
            occ_per_method[f"KL-IG ({sigma_name})"].append(0.0 if np.isnan(rho) else rho)

        for m in ["IDG", "ExpGrad", "IG-zero", "SmoothGrad", "Vanilla Grad"]:
            amap = COMPUTE_FN[m](model, x, tgt)
            ag = attr_to_grid(amap, ps, OCCLUSION_STRIDE, H, W)
            rho, _ = stats.spearmanr(om.ravel(), ag.ravel())
            occ_per_method[m].append(0.0 if np.isnan(rho) else rho)

    for m in methods_all:
        occ_by_ratio[m].append(np.mean(occ_per_method[m]))

fig, ax = plt.subplots(figsize=(13, 6))
ratio_pcts = [r * 100 for r in OCCLUSION_RATIOS]
for m in methods_all:
    ax.plot(ratio_pcts, occ_by_ratio[m], "o-", color=COLORS_ALL[m], lw=2, ms=6, label=m)
ax.set_xlabel("Occlusion ratio (%)", fontsize=11)
ax.set_ylabel("CC (Spearman ρ)", fontsize=11)
ax.set_title(f"Occlusion Correlation vs Ratio  n={len(eval_idx)}", fontweight="bold", fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Metric 4: Perturbation Robustness (All 8 Methods) ──

all_rankcorr = {m: {ns: [] for ns in PERTURBATION_SIGMAS} for m in methods_all}

for img_i, idx in enumerate(tqdm(eval_idx, desc="robustness")):
    r = results[idx]
    x, tgt = r["x"], r["target"]

    for sigma_name in ["adapt", "cap=1", "σ=0.25"]:
        sf = get_sigma_final(sigma_name, model, x, tgt)
        _, clean_amap = compute_klig(model, x, tgt, sf)
        clean_flat = clean_amap.detach().cpu().numpy().ravel()

        for ns in PERTURBATION_SIGMAS:
            rhos = []
            for _ in range(PERTURBATION_RUNS):
                _, pert_amap = compute_klig(model, x + torch.randn_like(x) * ns, tgt, sf)
                rho, _ = stats.spearmanr(clean_flat, pert_amap.detach().cpu().numpy().ravel())
                rhos.append(rho if not np.isnan(rho) else 0.0)
            all_rankcorr[f"KL-IG ({sigma_name})"][ns].append(np.mean(rhos))

    for m in ["IDG", "ExpGrad", "IG-zero", "SmoothGrad", "Vanilla Grad"]:
        clean_amap = COMPUTE_FN[m](model, x, tgt)
        clean_flat = clean_amap.detach().cpu().numpy().ravel()

        for ns in PERTURBATION_SIGMAS:
            rhos = []
            for _ in range(PERTURBATION_RUNS):
                pert_amap = COMPUTE_FN[m](model, x + torch.randn_like(x) * ns, tgt)
                rho, _ = stats.spearmanr(clean_flat, pert_amap.detach().cpu().numpy().ravel())
                rhos.append(rho if not np.isnan(rho) else 0.0)
            all_rankcorr[m][ns].append(np.mean(rhos))

fig, ax = plt.subplots(figsize=(13, 6))
for m in methods_all:
    rho_means = [np.mean(all_rankcorr[m][ns]) for ns in PERTURBATION_SIGMAS]
    ax.plot(PERTURBATION_SIGMAS, rho_means, "o-", color=COLORS_ALL[m], lw=2, ms=5,
            label=f"{m} ({np.mean(rho_means):.3f})")
ax.set_xlabel("Perturbation σ", fontsize=11)
ax.set_ylabel("Spearman ρ", fontsize=11)
ax.set_title(f"Perturbation Robustness (↑)  n={len(eval_idx)}", fontweight="bold", fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8, loc="best")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Metric 5: Object Focus Ratio (All 8 Methods) ──

all_ofr = {m: [] for m in methods_all}

for img_i, idx in enumerate(tqdm(eval_idx, desc="OFR")):
    r = results[idx]
    x = r["x"]
    _, klig_map = compute_klig(model, x, r["target"], get_sigma_final("adapt", model, x, r["target"]))
    obj_mask = estimate_object_mask(x, klig_map)

    for sigma_name in ["adapt", "cap=1", "σ=0.25"]:
        sf = get_sigma_final(sigma_name, model, x, r["target"])
        _, amap = compute_klig(model, x, r["target"], sf)
        all_ofr[f"KL-IG ({sigma_name})"].append(object_focus_ratio(amap, obj_mask))

    for m in ["IDG", "ExpGrad", "IG-zero", "SmoothGrad", "Vanilla Grad"]:
        amap = COMPUTE_FN[m](model, x, r["target"])
        all_ofr[m].append(object_focus_ratio(amap, obj_mask))

x_pos = np.arange(len(methods_all))
ofr_means = [np.mean(all_ofr[m]) for m in methods_all]
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x_pos, ofr_means, color=[COLORS_ALL[m] for m in methods_all], edgecolor="black", alpha=0.85)
for i, v in enumerate(ofr_means):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
ax.set_xticks(x_pos)
ax.set_xticklabels(methods_all, rotation=20, fontsize=9)
ax.set_ylabel("OFR", fontsize=11)
ax.set_title(f"Object Focus Ratio (↑)  n={len(eval_idx)}", fontweight="bold", fontsize=12)
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Metric 6: Gini Coefficient (Attribution Sparsity, All 8 Methods) ──

all_gini = {m: [] for m in methods_all}

for img_i, idx in enumerate(tqdm(eval_idx, desc="gini")):
    r = results[idx]
    x, tgt = r["x"], r["target"]

    for sigma_name in ["adapt", "cap=1", "σ=0.25"]:
        sf = get_sigma_final(sigma_name, model, x, tgt)
        _, amap = compute_klig(model, x, tgt, sf)
        all_gini[f"KL-IG ({sigma_name})"].append(gini_coefficient(amap))

    for m in ["IDG", "ExpGrad", "IG-zero", "SmoothGrad", "Vanilla Grad"]:
        amap = COMPUTE_FN[m](model, x, tgt)
        all_gini[m].append(gini_coefficient(amap))

x_pos = np.arange(len(methods_all))
gini_means = [np.mean(all_gini[m]) for m in methods_all]
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x_pos, gini_means, color=[COLORS_ALL[m] for m in methods_all], edgecolor="black", alpha=0.85)
for i, v in enumerate(gini_means):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
ax.set_xticks(x_pos)
ax.set_xticklabels(methods_all, rotation=20, fontsize=9)
ax.set_ylabel("Gini Coefficient", fontsize=11)
ax.set_title(f"Attribution Sparsity (↑)  n={len(eval_idx)}", fontweight="bold", fontsize=12)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Metric 7: Sanity Check (Trained vs Random Model, All 8 Methods) ──

model_random = copy.deepcopy(model)
torch.manual_seed(0)
model_random.fc.reset_parameters()
model_random.eval()

all_sanity = {m: [] for m in methods_all}

for img_i, idx in enumerate(tqdm(eval_idx, desc="sanity")):
    r = results[idx]
    x, tgt = r["x"], r["target"]

    for sigma_name in ["adapt", "cap=1", "σ=0.25"]:
        sf = get_sigma_final(sigma_name, model, x, tgt)
        _, amap_trained = compute_klig(model, x, tgt, sf)
        sf_rand = get_sigma_final(sigma_name, model_random, x, tgt)
        _, amap_random = compute_klig(model_random, x, tgt, sf_rand)
        all_sanity[f"KL-IG ({sigma_name})"].append(sanity_similarity(amap_trained, amap_random))

    for m in ["IDG", "ExpGrad", "IG-zero", "SmoothGrad", "Vanilla Grad"]:
        amap_trained = COMPUTE_FN[m](model, x, tgt)
        amap_random = COMPUTE_FN[m](model_random, x, tgt)
        all_sanity[m].append(sanity_similarity(amap_trained, amap_random))

x_pos = np.arange(len(methods_all))
sanity_means = [np.mean(all_sanity[m]) for m in methods_all]
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x_pos, sanity_means, color=[COLORS_ALL[m] for m in methods_all], edgecolor="black", alpha=0.85)
for i, v in enumerate(sanity_means):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
ax.axhline(0, color="gray", ls=":", alpha=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(methods_all, rotation=20, fontsize=9)
ax.set_ylabel("Spearman ρ (trained vs random)", fontsize=11)
ax.set_title(f"Sanity Check (↓ = faithful)  n={len(eval_idx)}", fontweight="bold", fontsize=12)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()
